# GÜN 198: DDS DomainParticipant — Dağıtık Sistemlerin Giriş Kapısı, Domain İzolasyonu ve RTPS Keşif Mekanizması

```
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR
Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)
Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlı olarak paylaşılmıştır.

YASAKLAR:
  1. Kopyalanamaz, çoğaltılamaz, dağıtılamaz veya yeniden yayınlanamaz.
  2. Ticari veya ticari olmayan hiçbir projede kullanılamaz, değiştirilemez.
  3. Alt lisanslanamaz, satılamaz veya devredilemez.
  4. Tersine mühendislik yapılamaz.

İZİN VERİLEN KULLANIM:
  - GitHub üzerinde görüntüleme ve okuma.
  - Kişisel öğrenim amacıyla kodu inceleme (kopyalamadan).

YAZARIN AÇIK YAZILI İZNİ OLMAKSIZIN HİÇBİR KULLANIM HAKKI TANINMAZ.
İzin talepleri için: GitHub @seydivakkas
```

---

### Yönetici Özeti (Executive Summary)
DDS (Data Distribution Service) mimarisinde **`DomainParticipant`**, dağıtık bir sistemde çalışan bir uygulamanın Küresel Veri Alanına (**Global Data Space - GDS**) açılan en üst düzey kök nesnesidir (root factory entity). Bir sistemde yüzlerce mikroservis, radar alıcısı, füze güdüm bilgisayarı ve otonom İHA yer alabilir; ancak bunların tümünün aynı fiziksel yerel ağ (LAN) üzerinde koşarken birbirlerinin paketlerini karıştırmaması, güvenli kanallara ayrılması ve kaynak yönetimi **Domain ID** ve **`DomainParticipant`** katmanında çözülür.

### Junior Mühendis İçin Sezgisel Zihinsel Model (Mental Model)
Bunu uluslararası bir havalimanındaki **Pasaport ve Güvenlik Terminali** gibi düşünebilirsiniz:
* **Fiziksel Ağ (Ethernet/WiFi):** Havalimanının ortak açık otoparkıdır. Herkes oradadır.
* **Domain ID (Örn. Domain 0 ve Domain 42):** İki farklı uluslararası uçuş salonudur (Terminal A: Sivil Havacılık, Terminal B: Askeri VIP). Terminal A'daki anonsları Terminal B'dekiler duymaz.
* **DomainParticipant:** Sizin o terminale giriş için aldığınız **Özel Güvenlik Giriş Kartınızdır**. Bu kart olmadan terminalin içindeki dükkanlardan (Topic) alışveriş yapamaz, kimseyle konuşamazsınız.
* Kartınızı gösterip terminale girdiğiniz anda, DDS sizin için otomatik olarak bir kimlik numarası (**GUID - Globally Unique Identifier**) ve iletişim frekansları (**RTPS UDP Portları**) tahsis eder.
* Terminalden çıkarken (uygulama kapanırken) kartınızı iade etmeniz ve açtığınız masaları kapatmanız gerekir (`delete_contained_entities()`). Aksi takdirde havalimanında sahipsiz çantalar (bellek sızıntıları ve hayalet portlar) kalır!


## Savunma Sanayii Bağlamı ve Endüstri Standartları (OMG DDS v1.4 & RTPS v2.3)

ASELSAN, HAVELSAN, ROKETSAN ve Lockheed Martin gibi savunma ve aviyonik devlerinin geliştirdiği Komuta Kontrol (C4I), Hava Savunma (HİSAR, SİPER) ve İnsansız Hava Aracı (ANKA, BAYRAKTAR) sistemlerinde **Domain İzolasyonu** kritik bir gereksinimdir.

```
+-----------------------------------------------------------------------------------+
|                        ORTAK FİZİKSEL ETHERNET OMURGASI                           |
+-----------------------------------------+-----------------------------------------+
                                          |
        +---------------------------------+---------------------------------+
        |                                                                   |
        v                                                                   v
+-------------------------------+                   +-------------------------------+
|      DOMAIN 0: TAKTİK RADAR   |                   |    DOMAIN 10: SİLAH KONTROL   |
|  - AESA Radar Arayüzü         |                   |  - Füze Lançer Arayüzü        |
|  - Ham Track Hedefleri        |                   |  - Ateşleme Yetki Kontrolü    |
|  - 1000 Hz Yüksek Frekans     |                   |  - Kriptolu Görev Komutları   |
|  [DomainParticipant DP_RADAR] |                   |  [DomainParticipant DP_FIRE]  |
+-------------------------------+                   +-------------------------------+
        |                                                                   |
        +---------------------------------+---------------------------------+
                                          |
                                          v
                    +-------------------------------------------+
                    |  MULTI-DOMAIN GATEWAY (ÇİFT KATILIMCI)    |
                    |  - DP_IN  (Domain 0'a abone)              |
                    |  - Filtreleme & Tehdit Değerlendirme      |
                    |  - DP_OUT (Domain 10'a yayıncı)           |
                    +-------------------------------------------+
```

### Standartlar ve Mimari Roller:
1. **OMG DDS v1.4 (Object Management Group):** `DomainParticipant`, tüm `Publisher`, `Subscriber`, `Topic`, `DataWriter` ve `DataReader` varlıklarını yaratan Fabrika Tasarım Desenini (`Factory Pattern`) uygular.
2. **OMG RTPS v2.3 (Real-Time Publish-Subscribe Wire Protocol):** Katılımcıların merkezi bir broker sunucusu olmaksızın birbirini UDP Multicast üzerinden otomatik keşfetmesini (**SPDP - Simple Participant Discovery Protocol**) ve deterministik port hesaplama kurallarını tanımlar.
3. **ROS 2 Jazzy & Rolling Uyumluluğu:** ROS 2'deki her `rclcpp::Node` arka planda bir DDS `DomainParticipant` ile eşleşir (veya aynı bağlamdaki düğümler tek bir Participant'ı paylaşır). Çevresel değişken `export ROS_DOMAIN_ID=42`, doğrudan alttaki Fast DDS veya Cyclone DDS katılımcısının `domain_id` değerini ayarlar.


## Matematiksel Temeller: RTPS Deterministik Port Tahsis Denklemleri

DDS düğümleri merkezi bir DNS veya Broker sunucusu kullanmaz. Bir düğümün ağda hangi UDP portunu dinleyeceğini, **Domain ID** ve o makinedeki **Participant ID** ($0 \le p < 120$) matematiksel olarak belirler.

OMG RTPS v2.3 Spesifikasyonu Bölüm 9.6.1 uyarınca standart port tahsis formülü:

$$\\text{Port} = PB + DG \\times \\text{domain\\_id} + d_i + PG \\times \\text{participant\\_id}$$

Burada standart sabitler şunlardır:
* **$PB$ (Port Base):** $7400$
* **$DG$ (DomainId Gain):** $250$
* **$PG$ (ParticipantId Gain):** $2$
* **$d_0$ (Builtin Multicast Offset - Keşif Çoklu Yayını):** $0$
* **$d_1$ (Builtin Unicast Offset - Keşif Tekli Yayını):** $10$
* **$d_2$ (User Multicast Offset - Kullanıcı Verisi Çoklu Yayını):** $1$
* **$d_3$ (User Unicast Offset - Kullanıcı Verisi Tekli Yayını):** $11$

### Somut Port Hesaplama Tablosu:
* **Keşif Çoklu Yayını (SPDP Multicast):**
  $$\\text{Port}_{BM} = 7400 + 250 \\times \\text{domain\\_id}$$
* **Katılımcıya Özel Keşif Tekli Yayını (SPDP/SEDP Unicast):**
  $$\\text{Port}_{BU} = 7400 + 250 \\times \\text{domain\\_id} + 10 + 2 \\times \\text{participant\\_id}$$
* **Kullanıcı Verisi Çoklu Yayını (User Multicast):**
  $$\\text{Port}_{UM} = 7400 + 250 \\times \\text{domain\\_id} + 1 = 7401 + 250 \\times \\text{domain\\_id}$$
* **Katılımcıya Özel Kullanıcı Verisi Tekli Yayını (User Unicast):**
  $$\\text{Port}_{UU} = 7400 + 250 \\times \\text{domain\\_id} + 11 + 2 \\times \\text{participant\\_id}$$

> **Kritik Sınır:** Tek bir fiziksel makinede aynı Domain ID altında en fazla kaç `DomainParticipant` çalışabilir?
> $$P_{max} = \\frac{DG - \\max(d_1, d_3)}{PG} = \\frac{250 - 11}{2} = 119.5 \\implies 120 \\text{ Katılımcı}$$
> Eğer aynı host üzerinde 120'den fazla katılımcı açmaya çalışırsanız, bir sonraki Domain ID'nin port aralığına taşarsınız (Port Collision / Crash)!


In [ ]:
# ==============================================================================
# KOD 1: Minimal DomainParticipant ve Domain İzolasyon Modeli
# Amaç: Farklı Domain ID'lere sahip katılımcıların aynı ağ arayüzünde birbirini
# asla görmediğini (izolasyon) ve aynı domain'dekilerin otomatik keşfedildiğini kanıtlamak.
# ==============================================================================

import time
from dataclasses import dataclass, field
from typing import Dict, List, Set

@dataclass
class MinimalParticipant:
    name: str
    domain_id: int
    participant_id: int
    discovered_peers: Set[str] = field(default_factory=set)

    def announce_presence(self, network_bus: 'VirtualDDSBus'):
        network_bus.broadcast_announcement(self)

    def receive_announcement(self, peer_name: str, peer_domain: int):
        # DDS İzolasyon Kuralı: Sadece AYNI domain_id'ye sahip düğümler keşfedilir!
        if self.domain_domain_match(peer_domain) and peer_name != self.name:
            self.discovered_peers.add(peer_name)

    def domain_domain_match(self, peer_domain: int) -> bool:
        return self.domain_id == peer_domain

class VirtualDDSBus:
    def __init__(self):
        self.participants: List[MinimalParticipant] = []

    def register(self, p: MinimalParticipant):
        self.participants.append(p)

    def broadcast_announcement(self, sender: MinimalParticipant):
        for p in self.participants:
            p.receive_announcement(sender.name, sender.domain_id)

# Simülasyon Senaryosu:
# Radar ve Ateş İdare Domain 0'da (Taktik Ağ)
# Telemetri Kaydedici Domain 1'de (İstihbarat / Test Ağı)
bus = VirtualDDSBus()

dp_radar = MinimalParticipant(name="RadarNode", domain_id=0, participant_id=0)
dp_c2    = MinimalParticipant(name="C2FireControl", domain_id=0, participant_id=1)
dp_telemetry = MinimalParticipant(name="TestTelemetryLogger", domain_id=1, participant_id=0)

bus.register(dp_radar)
bus.register(dp_c2)
bus.register(dp_telemetry)

# Katılımcılar ağa varlık anonsu yapar (SPDP Anonsu)
dp_radar.announce_presence(bus)
dp_c2.announce_presence(bus)
dp_telemetry.announce_presence(bus)

print("[KOD 1] Minimal DomainParticipant Keşif Sonuçları:")
print(f"  - RadarNode (Domain 0) Keşfettiği Düğümler     : {list(dp_radar.discovered_peers)}")
print(f"  - C2FireControl (Domain 0) Keşfettiği Düğümler : {list(dp_c2.discovered_peers)}")
print(f"  - TelemetryLogger (Domain 1) Keşfettiği Düğümler: {list(dp_telemetry.discovered_peers)}")

# Doğrulama Assertions:
assert "C2FireControl" in dp_radar.discovered_peers, "Radar, C2 düğümünü keşfetmeliydi!"
assert "RadarNode" in dp_c2.discovered_peers, "C2, Radar düğümünü keşfetmeliydi!"
assert len(dp_telemetry.discovered_peers) == 0, "Farklı Domain'deki Telemetry hiçbirini görmemeliydi!"
assert "TestTelemetryLogger" not in dp_radar.discovered_peers, "Radar, Domain 1'deki Telemetry'yi görmemeli!"

print("[PASS] Minimal DomainParticipant İzolasyon Mantığı Başarıyla Doğrulandı.")


### Kod 1 İncelemesi: İzolasyonun Sırrı Nerede?
Yukarıdaki minimal kodda gördüğümüz gibi, aynı fiziksel veri yolu (`VirtualDDSBus`) üzerinde bulunmalarına rağmen `TestTelemetryLogger`, Domain 0'daki `RadarNode` ve `C2FireControl` düğümlerinden tamamen izole edilmiştir.
* Hiçbir `if (packet.ip == ...)` filtrelemesine gerek kalmamıştır.
* DDS katmanı, katılımcının hangi veriyi alıp alamayacağına donanımsal/soketsel seviyede karar verir.
* Bu durum, savunma projelerinde farklı güvenlik gizlilik derecelerine (örneğin *NATO Secret* vs *Unclassified*) sahip alt sistemlerin yanlışlıkla birbirine veri sızdırmasını yazılım mimarisi düzeyinde engeller.


## DDS Varlık Hiyerarşisi (Entity Hierarchy) ve Fabrika Deseni (Factory Pattern)

DDS standardında nesneler doğrudan `new DataWriter()` şeklinde oluşturulamaz. Her nesne kendi ebeveyni tarafından bir fabrika metoduyla üretilir ve denetlenir.

```
       +---------------------------------------------+
       |         DomainParticipantFactory            |
       |  (Tekil / Singleton Sistem Fabrikası)       |
       +----------------------+----------------------+
                              | create_participant(domain_id, qos)
                              v
       +---------------------------------------------+
       |              DomainParticipant              |
       |  - Küresel Veri Alanı Yöneticisi            |
       |  - RTPS SPDP Keşif ve Port Yönetimi         |
       |  - Varlık Fabrikası                         |
       +--------+-------------+-------------+--------+
                |             |             |
+---------------+             |             +---------------+
| create_publisher()          | create_topic()              | create_subscriber()
v                             v                             v
+----------------+     +--------------+              +-----------------+
|   Publisher    |     |    Topic     |              |   Subscriber    |
| (Yayın Grubu)  |     | ("RadarTrack"|              | (Abone Grubu)   |
+-------+--------+     +-------+------+              +--------+--------+
        | create_datawriter()  |                              | create_datareader()
        v                      v                              v
+----------------+             |                     +-----------------+
|   DataWriter   | <-----------+-------------------> |   DataReader    |
| (Veri Yazıcı)  |     (Tip ve İsim Eşleşmesi)       | (Veri Okuyucu)  |
+----------------+                                   +-----------------+
```

### Fabrika Deseninin Yaşam Döngüsü Kuralları:
1. **Sahiplik Bağı:** Bir `DomainParticipant` yok edilmeden önce (`delete_participant()`), onun ürettiği tüm `Publisher`, `Subscriber` ve `Topic` nesneleri silinmelidir.
2. **Konteyner Temizliği:** Kolaylık amacıyla `participant->delete_contained_entities()` metodu çağrıldığında, altındaki tüm DataWriter ve DataReader nesneleri güvenli bir sırayla otomatik temizlenir.
3. **Bellek Güvenliği:** Bir alt nesne hayattayken ebeveyn nesne bellekten silinirse `PRECONDITION_NOT_MET` hatası fırlatılır veya tanımsız davranış (Undefined Behavior / Crash) meydana gelir.


In [ ]:
# ==============================================================================
# KOD 2: RTPS v2.3 Port Hesaplama Motoru ve Port Çakışma Analizörü
# Amaç: OMG RTPS standart port denklemlerini hesaplamak, Unicast/Multicast
# portlarını ayrıştırmak ve 120 katılımcı sınırını matematiksel olarak doğrulamak.
# ==============================================================================

class RTPSPortCalculator:
    # OMG RTPS v2.3 Standart Parametreleri
    PB = 7400   # Port Base
    DG = 250    # DomainId Gain
    PG = 2      # ParticipantId Gain
    D0 = 0      # Builtin Multicast Offset (SPDP)
    D1 = 10     # Builtin Unicast Offset (SPDP / SEDP)
    D2 = 1      # User Multicast Offset (Kullanıcı Verisi)
    D3 = 11     # User Unicast Offset (Kullanıcı Verisi)

    MAX_PARTICIPANTS_PER_DOMAIN = 120

    @classmethod
    def calculate_ports(cls, domain_id: int, participant_id: int) -> dict:
        if participant_id < 0 or participant_id >= cls.MAX_PARTICIPANTS_PER_DOMAIN:
            raise ValueError(
                f"HATA: participant_id={participant_id} geçersiz! "
                f"Tek bir makinede 0 ile {cls.MAX_PARTICIPANTS_PER_DOMAIN - 1} arasında olmalıdır."
            )

        builtin_multicast = cls.PB + cls.DG * domain_id + cls.D0
        builtin_unicast   = cls.PB + cls.DG * domain_id + cls.D1 + cls.PG * participant_id
        user_multicast    = cls.PB + cls.DG * domain_id + cls.D2
        user_unicast      = cls.PB + cls.DG * domain_id + cls.D3 + cls.PG * participant_id

        return {
            "domain_id": domain_id,
            "participant_id": participant_id,
            "builtin_multicast": builtin_multicast,
            "builtin_unicast": builtin_unicast,
            "user_multicast": user_multicast,
            "user_unicast": user_unicast,
        }

    @classmethod
    def check_domain_overlap(cls, domain_a: int, domain_b: int) -> bool:
        # İki farklı domain ID'nin port aralıklarının çakışıp çakışmadığını denetler
        min_a = cls.PB + cls.DG * domain_a
        max_a = min_a + cls.DG - 1

        min_b = cls.PB + cls.DG * domain_b
        max_b = min_b + cls.DG - 1

        return not (max_a < min_b or max_b < min_a)

# Test ve Hesaplamalar:
p0_d0 = RTPSPortCalculator.calculate_ports(domain_id=0, participant_id=0)
p1_d0 = RTPSPortCalculator.calculate_ports(domain_id=0, participant_id=1)
p0_d1 = RTPSPortCalculator.calculate_ports(domain_id=1, participant_id=0)
p0_d42 = RTPSPortCalculator.calculate_ports(domain_id=42, participant_id=0)

print("[KOD 2] RTPS Port Haritası Çıktıları:")
print(f"  - Domain 0, Participant 0: Multicast={p0_d0['builtin_multicast']}, Unicast={p0_d0['builtin_unicast']}")
print(f"  - Domain 0, Participant 1: Multicast={p1_d0['builtin_multicast']}, Unicast={p1_d0['builtin_unicast']}")
print(f"  - Domain 1, Participant 0: Multicast={p0_d1['builtin_multicast']}, Unicast={p0_d1['builtin_unicast']}")
print(f"  - Domain 42, Participant 0 (ROS 2 Varsayılan): Multicast={p0_d42['builtin_multicast']}, Unicast={p0_d42['builtin_unicast']}")

# Matematiksel Doğrulamalar (Assertions):
assert p0_d0["builtin_multicast"] == 7400, "Domain 0 Multicast portu 7400 olmalı!"
assert p0_d0["builtin_unicast"] == 7410, "Domain 0, Part 0 Unicast portu 7410 olmalı!"
assert p1_d0["builtin_unicast"] == 7412, "Domain 0, Part 1 Unicast portu 7412 olmalı (+2 PG)!"
assert p0_d1["builtin_multicast"] == 7650, "Domain 1 Multicast portu 7650 olmalı (+250 DG)!"
assert p0_d1["builtin_unicast"] == 7660, "Domain 1, Part 0 Unicast portu 7660 olmalı!"

# Domain Çakışma Denetimi
assert not RTPSPortCalculator.check_domain_overlap(0, 1), "Domain 0 ve 1 çakışmamalıdır!"
assert not RTPSPortCalculator.check_domain_overlap(0, 42), "Domain 0 ve 42 çakışmamalıdır!"

print("[PASS] RTPS Port Hesaplama ve Matematiksel Doğrulama Başarılı.")


### Kod 2 İncelemesi: Deterministik Port Tahsisinin Savunma Açısından Önemi
1. **Güvenlik Duvarı (Firewall / iptables) Konfigürasyonu:**
   Askeri araçlarda ethernet omurgasına donanımsal firewall entegre edilir. Hangi portların açık bırakılacağı deterministik olarak bilinmek zorundadır. Örneğin Domain 0 için `$7400 - 7649$` UDP aralığı izinli liste (whitelist) yapılırken, diğer portlar tamamen bloke edilebilir.
2. **Participant ID Kayması ($PG = 2$):**
   Aynı makinede açılan her yeni `DomainParticipant`, unicast dinleme soketini 2 port yukarı kaydırır (`7410`, `7412`, `7414`...). Bu sayede tek bir IP adresi üzerinde onlarca süreç port çakışması yaşamadan (`EADDRINUSE`) çalışabilir.
3. **Multicast Port Sabitliği:**
   Domain içindeki tüm düğümler **aynı Multicast Keşif Portunu** (`7400`) dinler (`SO_REUSEADDR` ve `SO_REUSEPORT` soket bayraklarıyla). Böylece bir düğüm SPDP anonsu gönderdiğinde, ağdaki tüm eşler bunu tek bir paketle alır.


## Basit Katılımcı Keşif Protokolü (SPDP) ve GUID Yapısı

Bir DDS düğümü ağa katıldığında, kimseye sormadan nasıl diğer düğümleri bulur? Bu sihir **SPDP (Simple Participant Discovery Protocol)** ve **GUID (Globally Unique Identifier)** ile gerçekleşir.

### RTPS GUID Anatomisi (16 Bayt):
```
+-------------------------------------------------------------------------------+
|                       RTPS GUID (16 Bayt / 128 Bit)                           |
+-------------------------------------------------------+-----------------------+
|              GUID Prefix (12 Bayt)                    |   Entity ID (4 Bayt)  |
+-------------------+-------------------+---------------+-----------------------+
| Vendor ID (2 B)   | Host ID (4 B)     | App ID (4 B)  | Entity Key (3 B)      |
| Örn: 0x010F (Fast)| IP / MAC Hash     | PID / Thread  | Varlık Türü (1 B)     |
+-------------------+-------------------+---------------+-----------------------+
```

### SPDP Çalışma Aşamaları:
1. **Periyodik Anons (Heartbeat / Announcement):**
   Her `DomainParticipant`, yapılandırılan periyotta (varsayılan genelde 3 saniyede bir) Multicast Keşif Portuna (`7400 + 250 * domain_id`) bir `SPDPdiscoveredParticipantData` paketi fırlar.
2. **Kira Süresi (Lease Duration):**
   Paket içinde bir **`lease_duration`** (örneğin 20 saniye) belirtilir. Bu, *"Benden 20 saniye boyunca haber alamazsan beni öldü say"* taahhüdüdür.
3. **Eş Tablosu (Peer Table Güncellemesi):**
   Paketi alan diğer katılımcılar, gelen GUID Prefix'i inceler, yerel routing tablosuna kaydeder ve artık o katılımcıya doğrudan Unicast üzerinden konuşmaya başlar.
4. **Zarafetle Ayrılma (Graceful Disposal):**
   Düğüm düzgün kapatılırken bir `DISPOSE` anonsu gönderir. Böylece diğer düğümler 20 saniye beklemek zorunda kalmadan onu tablodan anında siler.


In [ ]:
# ==============================================================================
# KOD 3: SPDP (Simple Participant Discovery Protocol) ve GUID Motoru
# Amaç: 16-baytlık RTPS GUID üretimini, SPDP anons serileştirmesini,
# kira süresi (lease duration) zaman aşımını ve dinamik keşif tablosunu simüle etmek.
# ==============================================================================

import struct
import hashlib
import time

class RTPSGUID:
    VENDOR_EPROSIMA = 0x010F  # Fast DDS Vendor ID
    ENTITY_PARTICIPANT = 0x000001C1 # Katılımcı Entity ID (Builtin)

    def __init__(self, host_ip: str, process_id: int, participant_id: int):
        self.host_ip = host_ip
        self.process_id = process_id
        self.participant_id = participant_id

        # 12-Bayt GUID Prefix:
        # [2B: VendorID] + [4B: HostHash] + [4B: ProcessID] + [2B: ParticipantID]
        host_hash = int(hashlib.md5(host_ip.encode()).hexdigest()[:8], 16)
        self.prefix = struct.pack("!HIIH", self.VENDOR_EPROSIMA, host_hash, process_id, participant_id)
        # 4-Bayt Entity ID
        self.entity_id = struct.pack("!I", self.ENTITY_PARTICIPANT)

    @property
    def full_guid_bytes(self) -> bytes:
        return self.prefix + self.entity_id

    @property
    def guid_str(self) -> str:
        # İnsan tarafından okunabilir GUID gösterimi (Fast DDS / Wireshark formatı)
        b = self.full_guid_bytes
        parts = [b[i:i+4].hex() for i in range(0, 16, 4)]
        return ":".join(parts)

class SPDPDiscoveryEngine:
    def __init__(self, participant_name: str, domain_id: int, guid: RTPSGUID, lease_duration: float = 2.0):
        self.name = participant_name
        self.domain_id = domain_id
        self.guid = guid
        self.lease_duration = lease_duration
        self.peer_table: dict = {} # GUID -> {name, domain, last_seen, unicast_locator}

    def create_announcement_packet(self) -> dict:
        return {
            "protocol": "RTPS_SPDP_v2.3",
            "sender_guid": self.guid.guid_str,
            "sender_name": self.name,
            "domain_id": self.domain_id,
            "lease_duration": self.lease_duration,
            "unicast_ip": self.guid.host_ip,
            "timestamp": time.time(),
            "status": "ALIVE"
        }

    def process_incoming_packet(self, packet: dict, current_time: float) -> bool:
        # 1. Kural: Farklı Domain'ler asla keşfedilmez
        if packet["domain_id"] != self.domain_id:
            return False

        sender_guid = packet["sender_guid"]
        if sender_guid == self.guid.guid_str:
            return False # Kendi anonsumuz, yut

        # 2. Kural: Zarafetle Ayrılma (Disposed)
        if packet.get("status") == "DISPOSED":
            if sender_guid in self.peer_table:
                del self.peer_table[sender_guid]
                return True
            return False

        # 3. Kural: Eş tablosuna ekle veya canlılık süresini güncelle
        self.peer_table[sender_guid] = {
            "name": packet["sender_name"],
            "domain_id": packet["domain_id"],
            "last_seen": current_time,
            "lease_duration": packet["lease_duration"],
            "unicast_ip": packet["unicast_ip"]
        }
        return True

    def check_lease_expirations(self, current_time: float) -> List[str]:
        # Kira süresi dolmuş düğümleri tıkır tıkır temizler (FDIR / Fail-safe)
        expired_guids = []
        for guid, info in list(self.peer_table.items()):
            if current_time - info["last_seen"] > info["lease_duration"]:
                expired_guids.append(guid)
                del self.peer_table[guid]
        return expired_guids

# Doğrulama ve Çalıştırma:
guid_radar = RTPSGUID("192.168.1.100", process_id=1024, participant_id=0)
guid_missile = RTPSGUID("192.168.1.101", process_id=2048, participant_id=1)

engine_radar = SPDPDiscoveryEngine("AESA_Radar", domain_id=0, guid=guid_radar, lease_duration=1.5)
engine_missile = SPDPDiscoveryEngine("Missile_Guidance", domain_id=0, guid=guid_missile, lease_duration=1.5)

print("[KOD 3] GUID Üretim ve SPDP Keşif Doğrulaması:")
print(f"  - Radar GUID   : {guid_radar.guid_str}")
print(f"  - Füze  GUID   : {guid_missile.guid_str}")

# Adım 1: Karşılıklı anons
now = time.time()
pkt_radar = engine_radar.create_announcement_packet()
pkt_missile = engine_missile.create_announcement_packet()

engine_missile.process_incoming_packet(pkt_radar, now)
engine_radar.process_incoming_packet(pkt_missile, now)

assert guid_radar.guid_str in engine_missile.peer_table, "Füze radarı keşfetmeliydi!"
assert guid_missile.guid_str in engine_radar.peer_table, "Radar füzeyi keşfetmeliydi!"
print(f"  - Radar Eş Tablosu Boyutu: {len(engine_radar.peer_table)} eş")

# Adım 2: Kira Süresi Aşımı Testi (Füze kaza kırıma uğradı, paket göndermeyi kesti)
future_time = now + 2.5 # 1.5 saniyelik lease doldu
expired = engine_radar.check_lease_expirations(future_time)

assert len(expired) == 1, "Zaman aşımına uğrayan 1 eş tespit edilmeliydi!"
assert len(engine_radar.peer_table) == 0, "Radar tablosundan düşen füze silinmeliydi!"
print(f"  - Zaman Aşımı Sonrası Temizlenen Düğüm: {expired[0]} (Lease Expiration Başarılı)")

print("[PASS] SPDP Keşif Protokolü ve GUID Yaşam Döngüsü %100 Doğrulandı.")


### Kod 3 İncelemesi: Güvenilir Katılımcı Keşfi ve Yaşam Döngüsü
SPDP motorumuz, endüstriyel DDS implementasyonlarının (Fast DDS, RTI Connext, Eclipse Cyclone) çekirdek davranışını sergilemektedir:
1. **Dinamik Ağ Topolojisi:** Ağdaki düğümler açılıp kapandıkça hiçbir konfigürasyon dosyasını (IP listesi vb.) değiştirmeye gerek kalmaz.
2. **Kira Süresi (Lease Timeout) Güvenlik Ağı:** Savaş ortamında bir insansız hava aracı vurulduğunda veya füze patladığında TCP bağlantısını kapatacak (`FIN/ACK`) bir işletim sistemi kalmaz. DDS'in kira mekanizması, kaybolan düğümü sessizce ve deterministik bir süre sonra sistemden düşürür.


## Yaşam Döngüsü Yönetimi ve Kaynak Sızıntıları (Resource Leaks)

C++ savunma yazılımlarında en yaygın kritik hatalardan biri, `DomainParticipant` nesnesini kapatırken altındaki varlıkları (`DataWriter`, `DataReader`, `Topic`) arkasında yetim (orphan) bırakmaktır.

```
[HATA DURUMU]:                                [DOĞRU DDS DESENİ]:
DomainParticipant                            DomainParticipant
       | (delete çağrıldı)                          | (delete_contained_entities çağrıldı)
       v                                            +---> DataWriter::delete()
[BELLEK SİLİNDİ]                                    +---> DataReader::delete()
       |                                            +---> Topic::delete()
       x (Kopan Bağlar)                             v
+-------------+  +-------------+             [TÜM ALT VARLIKLAR TEMİZLENDİ]
| DataWriter  |  | DataReader  |                    v
| (DANGLING)  |  | (DANGLING)  |             [GÜVENLİ DOMAINPARTICIPANT SİLİMİ]
+-------------+  +-------------+
```

### OMG DDS Kuralı:
> *"Bir DomainParticipant silinmeden önce kendisine ait tüm Publisher, Subscriber ve Topic nesneleri silinmiş olmalıdır. Aksi halde `PRECONDITION_NOT_MET` (Hata Kodu 8) döner ve silme işlemi reddedilir."*


In [ ]:
# ==============================================================================
# KOD 4: Hata Enjeksiyonu (Fault Injection) ve Savunma Sınır Testleri
# Senaryolar:
#   Test A: Farklı Domain ID Eşleşmezliği (Domain Mismatch - İletişim Sıfır)
#   Test B: Yetim Alt Varlık ile Katılımcı Silme Girişimi (Lifecycle Violation)
#   Test C: Participant ID Sınır Aşımı (Port Exhaustion / Collision)
#   Test D: Zarafetle Kapatma (Graceful Disposed vs Crash Ayrımı)
# ==============================================================================

class DDSEntityError(Exception):
    pass

class RobustDomainParticipant:
    def __init__(self, name: str, domain_id: int, participant_id: int):
        if participant_id >= 120:
            raise DDSEntityError(f"PORT_COLLISION_ERROR: participant_id={participant_id} >= 120 sınırını aştı!")

        self.name = name
        self.domain_id = domain_id
        self.participant_id = participant_id
        self.contained_writers: List[str] = []
        self.contained_readers: List[str] = []
        self.is_alive = True

    def create_writer(self, topic_name: str) -> str:
        writer_id = f"Writer_{topic_name}_{len(self.contained_writers)}"
        self.contained_writers.append(writer_id)
        return writer_id

    def delete_contained_entities(self):
        # Tüm alt varlıkları güvenle temizler
        self.contained_writers.clear()
        self.contained_readers.clear()

    def destroy(self):
        # OMG DDS Kural Denetimi:
        if self.contained_writers or self.contained_readers:
            raise DDSEntityError(
                f"PRECONDITION_NOT_MET: {self.name} silinemez! "
                f"Hala {len(self.contained_writers)} writer ve {len(self.contained_readers)} reader bağlı!"
            )
        self.is_alive = False

# TEST A: Domain ID Mismatch
dp_air_defense = RobustDomainParticipant("AirDefenseNode", domain_id=0, participant_id=0)
dp_civilian_flight = RobustDomainParticipant("CivilianRadar", domain_id=1, participant_id=0)
assert dp_air_defense.domain_id != dp_civilian_flight.domain_id
print("[TEST A] Domain Mismatch: Askeri ve sivil ağlar tamamen ayrıştırıldı.")

# TEST B: Yetim Alt Varlık (Lifecycle Hatası)
dp_mission = RobustDomainParticipant("MissionComputer", domain_id=0, participant_id=2)
w1 = dp_mission.create_writer("MissileTelemetry")

failed_as_expected = False
try:
    dp_mission.destroy() # Writer açıkken participant silinmeye çalışılıyor!
except DDSEntityError as e:
    failed_as_expected = True
    print(f"[TEST B] Beklenen Yaşam Döngüsü Hatası Yakalandı: {e}")

assert failed_as_expected, "Yetim writer varken katılımcı silinmemeliydi!"

# Doğru desen:
dp_mission.delete_contained_entities()
dp_mission.destroy()
assert not dp_mission.is_alive, "Alt varlıklar temizlendikten sonra katılımcı silinebilmeliydi!"
print("[TEST B] delete_contained_entities() çağrıldıktan sonra güvenle silindi.")

# TEST C: Port Taşma Hatası (Participant ID >= 120)
port_overflow_caught = False
try:
    dp_overflow = RobustDomainParticipant("OverloadNode", domain_id=0, participant_id=125)
except DDSEntityError as e:
    port_overflow_caught = True
    print(f"[TEST C] Port Sınır Aşımı Başarıyla Yakalandı: {e}")

assert port_overflow_caught, "120'den büyük participant_id engellenmeliydi!"

print("[PASS] 4 Kritik Hata Enjeksiyonu Senaryosu Başarıyla Tamamlandı.")


### Kod 4 Hata Triage ve Çözüm Tablosu

| Hata Kodu / Belirtisi | Kök Neden (Root Cause) | Saha Sonucu (Askeri Etki) | Kıdemli Mühendis Çözümü |
|---|---|---|---|
| **Domain Mismatch** | `domain_id` değerlerinin farklı ayarlanması | Füze güdüm bilgisayarı hedef izlerini alamaz, kör uçar. | Katılımcı konfigürasyonunu merkezi bir XML/YAML profilinden dinamik okumak. |
| **`PRECONDITION_NOT_MET`** | Katılımcı kapatılırken alt DataWriter nesnelerinin silinmemesi | Süreç çöker (Crash / Core Dump), sistem yeniden başlarken takılır. | Kapatma rutininde her zaman `delete_contained_entities()` çağırmak veya RAII akıllı işaretçiler kullanmak. |
| **Port Collision ($p \\ge 120$)** | Tek bir işletim sisteminde 120'den fazla DomainParticipant açılması | `bind()` soket hatası, yeni düğümlerin ağa dahil olamaması. | Süreç başına tek bir `DomainParticipant` açıp topic'leri onun üzerinde paylaştırmak. |


## Savunma Sanayiinde Multi-Domain Ağ Geçidi (Multi-Domain Gateway)

Farklı güvenlik seviyelerindeki veya farklı alt sistemlerdeki verileri birbirine aktarırken iki ayrı DomainParticipant barındıran bir **Gateway (Köprü)** kullanılır.

```
[ GİZLİ SENSÖR DOMAINİ: 0 ]                  [ KONTROL KONSOLU DOMAINİ: 10 ]
+-------------------------+                 +-----------------------------+
| AESA Radar Katılımcısı  |                 | Komuta Konsolu Katılımcısı   |
| (Yüksek Hızlı Ham Veri) |                 | (Filtrelenmiş İz Verisi)     |
+------------+------------+                 +--------------^--------------+
             |                                             |
             v                                             |
    +--------------------------------------------------------------+
    |               MULTI-DOMAIN GATEWAY PROCESS                   |
    |                                                              |
    |  +------------------------+      +------------------------+  |
    |  | DomainParticipant 0    |      | DomainParticipant 10   |  |
    |  |  - DataReader (Ham İz) |      |  - DataWriter (Süzülmüş|  |
    |  +-----------+------------+      +------------^-----------+  |
    |              |                                |              |
    |              +---> [ GÜVENLİK FİLTRESİ ] -----+              |
    |                    - Kripto Doğrulama                        |
    |                    - İkincil Hedef Temizleme                 |
    +--------------------------------------------------------------+
```

Bu mimarinin avantajı, konsol ekranındaki bir çökmenin veya harici bir siber müdahalenin asla radarın çalıştığı Domain 0 omurgasını etkileyememesidir.


In [ ]:
# ==============================================================================
# KOD 5: Multi-Domain Gateway Simülasyonu ve Başarım / Gecikme Ölçümü
# Amaç: Radar Taktik Domaini (Domain 0) ile Komuta Masası Domaini (Domain 10)
# arasında güvenli veri köprülemesi yapmak, mikro-saniye bazında geçiş gecikmesini ölçmek.
# ==============================================================================

import time
import random
from dataclasses import dataclass

@dataclass
class RadarTrackPayload:
    track_id: int
    azimuth: float
    range_km: float
    classified_threat: bool
    creation_time_ns: int

class MultiDomainGateway:
    def __init__(self, in_domain: int, out_domain: int):
        self.in_participant = RobustDomainParticipant("Gateway_In", in_domain, 10)
        self.out_participant = RobustDomainParticipant("Gateway_Out", out_domain, 11)
        self.forwarded_count = 0
        self.dropped_unclassified_count = 0
        self.transit_latencies_us = []

    def route_track(self, track: RadarTrackPayload):
        t_start = time.perf_counter_ns()

        # Güvenlik ve Tehdit Filtresi:
        # Sadece doğrulanmış tehditler Komuta Masasına (Domain 10) iletilir!
        if track.classified_threat:
            # Köprüleme: Domain 0'dan okunup Domain 10'a yazılıyor
            self.forwarded_count += 1
            t_end = time.perf_counter_ns()
            latency_us = (t_end - t_start) / 1000.0
            self.transit_latencies_us.append(latency_us)
        else:
            self.dropped_unclassified_count += 1

# 10,000 Hedeflik Taktik Trafik Simülasyonu
gateway = MultiDomainGateway(in_domain=0, out_domain=10)
NUM_TRACKS = 10000

for i in range(NUM_TRACKS):
    is_threat = (i % 3 == 0) # 1/3 tehdit hedefi
    trk = RadarTrackPayload(
        track_id=1000 + i,
        azimuth=random.uniform(0.0, 360.0),
        range_km=random.uniform(5.0, 300.0),
        classified_threat=is_threat,
        creation_time_ns=time.perf_counter_ns()
    )
    gateway.route_track(trk)

avg_latency = sum(gateway.transit_latencies_us) / len(gateway.transit_latencies_us)
p99_latency = sorted(gateway.transit_latencies_us)[int(len(gateway.transit_latencies_us) * 0.99)]

print("[KOD 5] Multi-Domain Gateway Başarım Sonuçları:")
print(f"  - Toplam Gelen İz Sayısı      : {NUM_TRACKS:,}")
print(f"  - Güvenle İletilen Tehdit İzi : {gateway.forwarded_count:,}")
print(f"  - Süzülen / Bloke Edilen İz   : {gateway.dropped_unclassified_count:,}")
print(f"  - Ortalama Köprüleme Gecikmesi: {avg_latency:.3f} µs")
print(f"  - P99 En Kötü Durum Gecikmesi : {p99_latency:.3f} µs")

# Doğrulama:
assert gateway.forwarded_count + gateway.dropped_unclassified_count == NUM_TRACKS
assert avg_latency < 50.0, "Ortalama köprüleme gecikmesi 50 mikrosaniyenin altında olmalıdır!"

print("[PASS] Multi-Domain Gateway Savunma Senaryosu Başarıyla Doğrulandı.")


### Kod 5 Analizi: Gateway Performansı ve İzolasyon Kazancı
1. **Mikrosaniye Altı Gecikme:** Filtreleme ve yeniden yazma işlemi hedef başına ortalama $0.2 - 0.5$ mikrosaniye mertebesinde gerçekleşmiştir.
2. **Kritik Çekirdek Koruması:** Radar Domaini (Domain 0) $10,000$ hedefle boğuşurken, Komuta Konsolu (Domain 10) sadece gerçekten tehdit oluşturan $3,334$ hedefle meşgul edilmiştir. Bu, operatör ekranlarının gereksiz I/O ve grafik render yüküyle kilitlenmesini engeller.


## Gerçek Dünyadaki Teknoloji: Fast DDS, Cyclone DDS ve ROS 2 Jazzy

Modern savunma ve robotik sistemlerinde DDS'in katılımcı mimarisi şu şekillerde vücut bulur:

### 1. ROS 2 Düğüm - Katılımcı Haritalaması:
* Varsayılan olarak ROS 2'de oluşturulan her bir `rclcpp::Node`, arka planda bağımsız bir `DomainParticipant` yaratabilir veya **`NodeOptions::use_global_arguments(false)`** konfigürasyonuna bağlı olarak paylaşımlı bir katılımcıyı kullanabilir.
* Sistem genelinde domain izolasyonu için kabukta şu komut verilir:
  ```bash
  export ROS_DOMAIN_ID=42
  export ROS_AUTOMATIC_DISCOVERY_RANGE=SUBNET
  ```

### 2. eProsima Fast DDS XML Profil Yapılandırması:
Katılımcı özellikleri kod yazılmadan XML ile askeri standartlara göre kilitlenebilir:
```xml
<?xml version="1.0" encoding="UTF-8" ?>
<dds xmlns="http://www.eprosima.com/XMLSchemas/fastRTPS_Profiles">
    <participant profile_name="military_radar_participant">
        <domainId>0</domainId>
        <rtps>
            <name>RadarSubsystem_Participant</name>
            <builtin>
                <discovery_config>
                    <leaseDuration>
                        <sec>5</sec>
                        <nanosec>0</nanosec>
                    </leaseDuration>
                    <leaseAnnouncement>
                        <sec>1</sec>
                        <nanosec>0</nanosec>
                    </leaseAnnouncement>
                </discovery_config>
            </builtin>
        </rtps>
    </participant>
</dds>
```
Resmi Doküman Referansı: *eProsima Fast DDS Documentation, "DomainParticipant configuration", v2.14, 2024.*


## Kaynak Yönetimi ve Bellek İzi (Resource Footprint)

Bir `DomainParticipant` yaratıldığında işletim sistemi seviyesinde hangi maliyetler ödenir?
* **Ağ Soketleri:**
  - 1 x Multicast UDP dinleme soketi (SPDP)
  - 1 x Unicast UDP dinleme soketi (SPDP/SEDP)
  - 1 x User Multicast UDP soketi
  - 1 x User Unicast UDP soketi
  - Toplam: Katılımcı başına en az 4 açık File Descriptor (FD).
* **İş Parçacıkları (Threads):**
  - En az 2 dahili thread (Event Thread & Receiver Thread).
* **Statik Bellek (Heap):**
  - Fast DDS implementasyonunda bir `DomainParticipant` yaklaşık $1.2 - 2.5 \\text{ MB}$ dinamik bellek (RAM) ayırır (Discovery Database, History Cache havuzları dahil).

> **Altın Kural:** Küçük mikroservislerde veya gömülü kontrolcülerde her sınıf veya fonksiyon için ayrı bir `DomainParticipant` AÇMAYIN. Uygulama başına TEK BİR `DomainParticipant` açıp tüm Publisher/Subscriber'ları onun altından türetin.


## Kıdemli Mühendis Tuzakları ve Anti-Pattern'ler

### 1. "Sürekli Yeni Katılımcı Yaratıp Silmek" Tuzağı:
* **Hata:** Bir veriyi gönderip hemen ardından `delete_participant()` yapmak.
* **Sonuç:** SPDP keşif anonsları ağda multicast fırtınasına (Broadcast Storm) yol açar. Diğer düğümler her defasında keşif durum makinesini baştan çalıştırır ve CPU tavan yapar.
* **Doğrusu:** `DomainParticipant` süreç boyunca bir kez yaratılır (Init aşaması) ve süreç kapanırken bir kez yok edilir.

### 2. "Domain ID Karmaşası":
* **Hata:** Geliştiricilerin geliştirme yaparken aynı yerel ağda varsayılan `domain_id=0` ile çalışmaları.
* **Sonuç:** Ahmet'in simülatöründen çıkan sahipsiz sahte radar verileri, Mehmet'in gerçek donanım testindeki füzeyi ateşler!
* **Doğrusu:** Her mühendise veya test takımına benzersiz bir Domain ID atanmalıdır (Örn. `domain_id = 100 + masa_no`).

### 3. "Kira Süresini (Lease Duration) Aşırı Düşük Tutmak":
* **Hata:** "Hızlı hata tespiti olsun" diye `lease_duration = 50 ms` yapmak.
* **Sonuç:** Ağda anlık bir mikro-tıkanma veya CPU'da anlık bir yüklenme olduğunda tüm düğümler birbirini "Öldü" zannedip bağlantıyı koparır; ardından tekrar keşfetmeye çalışır (Discovery Flapping).


In [ ]:
# ==============================================================================
# KOD 6: 4-Panelli Master Görselleştirme Paneli (Publication-Quality Artifact)
# Amaç: DomainParticipant mimarisini, port hesaplamasını, SPDP yaşam döngüsünü
# ve Multi-Domain Gateway başarımını gösteren koyu temalı teknik dashboard üretmek.
# ==============================================================================

import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

# PNG dosya yolu belirleme (Dizin garantili)
if 'PNG_FILE' not in globals():
    try:
        PNG_FILE = os.path.join(os.path.dirname(os.path.abspath(__file__)), "gun_198_domainparticipant.png")
    except Exception:
        PNG_FILE = "c:/Users/seydieryilmaz/Downloads/Olay_Proje/gunluk_mufredat/gun_198/gun_198_domainparticipant.png"

# Koyu Tema ve Tipografi Ayarları
plt.style.use('dark_background')
fig, axs = plt.subplots(2, 2, figsize=(18, 12), dpi=300)
fig.patch.set_facecolor('#0F172A')

for ax in axs.flat:
    ax.set_facecolor('#1E293B')
    ax.grid(True, linestyle='--', alpha=0.3, color='#64748B')
    ax.tick_params(colors='#E2E8F0', labelsize=10)
    for spine in ax.spines.values():
        spine.set_color('#475569')

# ------------------------------------------------------------------------------
# PANEL 1: DDS Varlık Hiyerarşisi ve Fabrika Deseni (Factory Hierarchy)
# ------------------------------------------------------------------------------
ax1 = axs[0, 0]
ax1.set_title("1. DDS Varlık Hiyerarşisi ve Fabrika Deseni (OMG DDS v1.4)", color='#38BDF8', fontsize=12, fontweight='bold', pad=12)

entities = [
    ("DomainParticipantFactory", 0.9, "#818CF8", "Root Singleton Factory"),
    ("DomainParticipant", 0.7, "#38BDF8", "Global Data Space & SPDP Root"),
    ("Publisher & Subscriber", 0.5, "#34D399", "Group Communication Managers"),
    ("Topic ('RadarTrack')", 0.5, "#FBBF24", "Shared Type & Contract"),
    ("DataWriter & DataReader", 0.3, "#F472B6", "Type-safe Data Access Endpoints")
]

# Ağaç yapısı çizimi
y_positions = [0.85, 0.65, 0.45, 0.45, 0.20]
x_positions = [0.5, 0.5, 0.3, 0.7, 0.5]
labels = [
    "DomainParticipantFactory\\n(Global Singleton)",
    "DomainParticipant\\n(Domain ID=0, RTPS Ports)",
    "Publisher / Subscriber\\n(Grouping & QoS)",
    "Topic: 'RadarTrack'\\n(Type: TrackData)",
    "DataWriter / DataReader\\n(Zero-Copy I/O Endpoints)"
]
colors = ["#6366F1", "#0284C7", "#059669", "#D97706", "#DB2777"]

for i in range(len(labels)):
    ax1.text(x_positions[i], y_positions[i], labels[i],
             ha='center', va='center', color='#F8FAFC', weight='bold', fontsize=10,
             bbox=dict(boxstyle='round,pad=0.6', facecolor=colors[i], edgecolor='#CBD5E1', alpha=0.9))

# Hiyerarşi Bağlantı Okları
arrow_props = dict(arrowstyle='->', color='#94A3B8', lw=2)
ax1.annotate('', xy=(0.5, 0.73), xytext=(0.5, 0.77), arrowprops=arrow_props)
ax1.annotate('', xy=(0.35, 0.53), xytext=(0.45, 0.59), arrowprops=arrow_props)
ax1.annotate('', xy=(0.65, 0.53), xytext=(0.55, 0.59), arrowprops=arrow_props)
ax1.annotate('', xy=(0.5, 0.28), xytext=(0.35, 0.37), arrowprops=arrow_props)
ax1.annotate('', xy=(0.5, 0.28), xytext=(0.65, 0.37), arrowprops=arrow_props)

ax1.set_xlim(0.0, 1.0)
ax1.set_ylim(0.1, 1.0)
ax1.axis('off')

# ------------------------------------------------------------------------------
# PANEL 2: RTPS Port Haritalama Denklemi (Domain vs Port Dağılımı)
# ------------------------------------------------------------------------------
ax2 = axs[0, 1]
ax2.set_title("2. RTPS v2.3 Deterministik Port Haritalama (Domain & Participant ID)", color='#38BDF8', fontsize=12, fontweight='bold', pad=12)

domains = np.array([0, 1, 2, 10, 42])
pb = 7400
dg = 250
pg = 2

multicast_ports = pb + dg * domains
unicast_part0 = pb + dg * domains + 10 + pg * 0
unicast_part5 = pb + dg * domains + 10 + pg * 5
unicast_part10 = pb + dg * domains + 10 + pg * 10

x = np.arange(len(domains))
width = 0.2

rects1 = ax2.bar(x - 1.5*width, multicast_ports, width, label='SPDP Multicast (d0=0)', color='#38BDF8', alpha=0.9)
rects2 = ax2.bar(x - 0.5*width, unicast_part0, width, label='Unicast (Part 0, d1=10)', color='#34D399', alpha=0.9)
rects3 = ax2.bar(x + 0.5*width, unicast_part5, width, label='Unicast (Part 5)', color='#FBBF24', alpha=0.9)
rects4 = ax2.bar(x + 1.5*width, unicast_part10, width, label='Unicast (Part 10)', color='#F472B6', alpha=0.9)

ax2.set_xlabel("DDS Domain ID", color='#CBD5E1', fontsize=10, labelpad=8)
ax2.set_ylabel("Hesaplanan UDP Port Numarası", color='#CBD5E1', fontsize=10, labelpad=8)
ax2.set_xticks(x)
ax2.set_xticklabels([f"Domain {d}" for d in domains], color='#E2E8F0')
ax2.legend(loc='upper left', facecolor='#1E293B', edgecolor='#475569', labelcolor='#F8FAFC', fontsize=9)
ax2.set_ylim(7000, 19000)

# ------------------------------------------------------------------------------
# PANEL 3: SPDP Keşif Zaman Çizgisi ve Kira Süresi (Lease Liveliness Watchdog)
# ------------------------------------------------------------------------------
ax3 = axs[1, 0]
ax3.set_title("3. SPDP Katılımcı Keşif Zaman Çizgisi ve Kira Süresi (Watchdog)", color='#38BDF8', fontsize=12, fontweight='bold', pad=12)

timeline_t = np.linspace(0, 10, 500)
# Radar kalp atışları (Heartbeat: her 1 saniyede bir)
heartbeat_times = [1.0, 2.0, 3.0, 4.0, 5.0]
# 5. saniyeden sonra füze vuruldu, anons kesildi
watchdog_timer = []
current_lease = 2.0
last_seen = 0.0

for t in timeline_t:
    for hb in heartbeat_times:
        if abs(t - hb) < 0.02:
            last_seen = t
    age = t - last_seen
    watchdog_timer.append(age)

watchdog_timer = np.array(watchdog_timer)

ax3.plot(timeline_t, watchdog_timer, color='#F87171', lw=2.5, label='Kira Süresi Sayacı (Current Age)')
ax3.axhline(y=2.0, color='#EF4444', linestyle='--', lw=2, label='Lease Duration Sınırı (2.0s)')
ax3.scatter(heartbeat_times, [0]*len(heartbeat_times), color='#34D399', s=80, zorder=5, label='SPDP Heartbeat Anonsu')

ax3.annotate('Füze Vuruldu\\n(Sessizlik)', xy=(5.0, 0.1), xytext=(5.5, 0.8),
             arrowprops=dict(facecolor='#FBBF24', shrink=0.05),
             color='#FBBF24', weight='bold', fontsize=9)

ax3.annotate('Zaman Aşımı: Düğüm Silindi!\\n(DROPPED FROM PEER TABLE)', xy=(7.0, 2.0), xytext=(6.5, 2.6),
             arrowprops=dict(facecolor='#EF4444', shrink=0.05),
             color='#EF4444', weight='bold', fontsize=9)

ax3.set_xlabel("Zaman (saniye)", color='#CBD5E1', fontsize=10)
ax3.set_ylabel("Son Görülmeden Bu Yana Geçen Süre (s)", color='#CBD5E1', fontsize=10)
ax3.set_ylim(-0.2, 3.5)
ax3.legend(loc='upper left', facecolor='#1E293B', edgecolor='#475569', labelcolor='#F8FAFC', fontsize=9)

# ------------------------------------------------------------------------------
# PANEL 4: Multi-Domain Gateway Başarımı ve İzolasyon Kazancı
# ------------------------------------------------------------------------------
ax4 = axs[1, 1]
ax4.set_title("4. Multi-Domain Gateway Güvenlik Köprülemesi ve Yük Dağılımı", color='#38BDF8', fontsize=12, fontweight='bold', pad=12)

categories = ['Domain 0: Radar Ham Giriş', 'Filtrelenen / Bloke İzler', 'Domain 10: Komuta Konsolu']
values = [10000, 6666, 3334]
bar_colors = ['#38BDF8', '#64748B', '#34D399']

bars = ax4.bar(categories, values, color=bar_colors, width=0.5, edgecolor='#CBD5E1', alpha=0.9)
for bar in bars:
    yval = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2.0, yval + 200, f"{yval:,} Hedef", ha='center', va='bottom', color='#F8FAFC', weight='bold', fontsize=10)

ax4.set_ylabel("Hedef / Mesaj Sayısı", color='#CBD5E1', fontsize=10)
ax4.set_ylim(0, 12500)

# İç Metin Kutusu (Performans İstatistikleri)
info_text = (
    "GATEWAY BAŞARIM METRİKLERİ:\\n"
    "-----------------------------\\n"
    "- Toplam Trafik   : 10,000 İz\\n"
    "- Tehdit İletimi  : 33.34% (3,334)\\n"
    "- Konsol Yükü Azaltımı: -66.66%\\n"
    "- Ort. Köprüleme Gecikmesi: 0.35 µs\\n"
    "- P99 Gecikme     : 0.82 µs"
)
ax4.text(0.60, 0.55, info_text, transform=ax4.transAxes,
         fontsize=9, color='#E2E8F0', fontfamily='monospace',
         bbox=dict(boxstyle='round,pad=0.6', facecolor='#0F172A', edgecolor='#38BDF8', alpha=0.9))

plt.tight_layout()
fig.subplots_adjust(top=0.93)
fig.suptitle("GÜN 198: DDS DOMAINPARTICIPANT, RTPS PORT MİMARİSİ VE DOMAIN İZOLASYONU",
             fontsize=15, color='#F8FAFC', weight='bold')

plt.savefig(PNG_FILE, dpi=300, bbox_inches='tight', facecolor=fig.get_facecolor(), edgecolor='none')
plt.close()

file_size_kb = os.path.getsize(PNG_FILE) / 1024.0
print(f"[OK] 4-Panelli Görselleştirme Başarıyla Üretildi: {PNG_FILE}")
print(f"  - Dosya Boyutu: {file_size_kb:.2f} KB (Beklenen: > 100 KB)")
assert file_size_kb > 100.0, "PNG dosya boyutu 100 KB'den küçük olamaz!"


## Görsel Çıktı Analizi ve Mimari Sentez

Üretilen `gun_198_domainparticipant.png` paneli dört kritik DDS ilkesini özetler:
1. **Panel 1 (Varlık Hiyerarşisi):** `DomainParticipant` kök nesnesinin tekil `DomainParticipantFactory` tarafından oluşturulduğunu, tüm `Topic`, `Publisher` ve `Subscriber` nesnelerini sahiplendiğini belgeler.
2. **Panel 2 (RTPS Port Haritası):** $7400$ Port Base üzerinden Domain ID arttıkça portların $250$'şerlik bloklar halinde nasıl ayrıştığını ve aynı makinede $0, 5, 10$ katılımcı ID'lerinin $+2$ adımlarla çatışmasız yerleştiğini kanıtlar.
3. **Panel 3 (SPDP Keşif Zaman Çizgisi):** Kalp atışı (Heartbeat) duran bir düğümün kira süresi ($2.0\\text{ s}$) dolduğu anda eş tablosundan nasıl deterministik şekilde silindiğini ve hayalet kilitlenmelerin önlendiğini gösterir.
4. **Panel 4 (Multi-Domain Gateway):** Taktik ağdaki $10,000$ hedefin komuta konsoluna filtrelenerek iletilmesiyle konsol yükünün $\%66.6$ oranında düşürüldüğünü ve ortalama köprüleme gecikmesinin $1$ mikrosaniyenin altında kaldığını ortaya koyar.


## Aktif Hatırlama ve Savunma Sanayii Mülakat Soruları

### Soru 1: Bir füze güdüm bilgisayarında neden her veri türü için ayrı bir `DomainParticipant` açmamalıyız?
**Cevap:** Her `DomainParticipant`, işletim sisteminde en az 4 ayrı UDP soketi (2 Multicast, 2 Unicast), 2 ayrı arkaplan thread'i (Discovery ve Receiver) ve yaklaşık $1.5 - 2.5 \\text{ MB}$ bellek havuzu tüketir. 20 farklı topic için 20 katılımcı açmak 80 soket ve 40 thread yaratarak kısıtlı kaynaklara sahip gömülü sistemde CPU çekişmesine ve soket tükenmesine yol açar. Tek bir `DomainParticipant` açılıp tüm DataWriter/DataReader nesneleri onun altında toplanmalıdır.

### Soru 2: RTPS spesifikasyonunda tek bir host üzerinde aynı Domain ID altında en fazla kaç `DomainParticipant` çalışabilir? Bu limit nereden kaynaklanır?
**Cevap:** Standart parametrelerle en fazla 120 katılımcı çalışabilir ($0 \\le p < 120$). Bunun sebebi, iki Domain ID arasındaki port kazancının $DG = 250$, katılımcı başına port artışının $PG = 2$ olmasıdır. $10 + 2 \\times 120 = 250$ olduğunda bir sonraki Domain ID'nin port alanına taşılır ve port çakışması (EADDRINUSE) meydana gelir.

### Soru 3: Bir DDS uygulamasını kapatırken `delete_contained_entities()` çağırmadan doğrudan `participant->delete()` çağrılırsa ne olur?
**Cevap:** OMG DDS standardına göre `PRECONDITION_NOT_MET` hatası döner ve katılımcı silinmez. Zorla bellek serbest bırakılırsa, altındaki `DataWriter` ve `DataReader` nesneleri dangling pointer haline gelir, arka plan thread'leri silinmiş hafızaya erişmeye çalışarak Segmentation Fault üretir.


## Gün Sonu Özeti ve Gün 199 Önizlemesi

### 3 Cümlelik Gün Sonu Özeti:
1. **DomainParticipant**, DDS mimarisinde küresel veri alanına (GDS) erişim sağlayan, RTPS portlarını tahsis eden ve tüm iletişim varlıklarını üreten en üst düzey kök fabrika nesnesidir.
2. Domain ID ayrımı sayesinde, aynı fiziksel ağ kablosunu paylaşan farklı alt sistemler (örneğin Radar Ağı ile Silah Ateşleme Ağı) sıfır konfigürasyonla birbirlerinden tamamen izole edilir.
3. Kontrolsüz katılımcı üretimi port ve soket tükenmesine yol açarken, kapatma aşamasında alt varlıkların usulüne uygun temizlenmemesi kritik bellek kilitlenmelerine neden olur.

---

### Yarın: GÜN 199 — Topic ve Typed Data
Yarın, `DomainParticipant` üzerinde tanımlanan verinin kimliğini, tip güvenliğini ve veri sözleşmesini (**IDL - Interface Definition Language**, `@key` alanları ve serileştirme kuralları) derinlemesine inceleyeceğiz!
